# GPPO Phase-1 Formal Matrix (Colab)

Runs the frozen four-scale, five-seed formal matrix on Colab and resumes the migrated local checkpoints. Outputs are backed up to Google Drive every five minutes. Do not change architecture or formal hyperparameters after test results are read.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys, time, json
from pathlib import Path
REPO = Path('/content/GPPO')
BRANCH = '8.8-GPPO无偏好'
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/Battleplus/GPPO.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements.txt')], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
print(torch.cuda.get_device_name(0))
print('commit', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/GPPO_phase1_formal')
MIGRATION_ZIP = Path('/content/drive/MyDrive/GPPO_phase1_migration.zip')
LOCAL_ROOT = Path('/content/phase1_formal')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
if not LOCAL_ROOT.exists():
    LOCAL_ROOT.mkdir(parents=True)
    if MIGRATION_ZIP.exists():
        shutil.unpack_archive(str(MIGRATION_ZIP), str(LOCAL_ROOT))
    elif any(DRIVE_ROOT.iterdir()):
        shutil.copytree(DRIVE_ROOT, LOCAL_ROOT, dirs_exist_ok=True)
    else:
        raise FileNotFoundError(f'Missing migration archive: {MIGRATION_ZIP}')
print('local state', LOCAL_ROOT)
print('drive backup', DRIVE_ROOT)


In [ ]:
# Formal protocol is frozen. Only JOBS is a resource-control setting.
JOBS = 2
cmd = [sys.executable, str(REPO/'run_phase1_formal_matrix.py'),
       '--frozen-protocol', str(REPO/'configs/PHASE1_FROZEN_PROTOCOL.json'),
       '--formal-root', str(LOCAL_ROOT),
       '--legacy-literal-root', str(LOCAL_ROOT/'T5-10-48_literal_event'),
       '--python', sys.executable, '--jobs', str(JOBS), '--device', 'cuda']
log_path = DRIVE_ROOT/'colab_phase1_launcher.log'
log_stream = log_path.open('a', encoding='utf-8')
process = subprocess.Popen(cmd, cwd=REPO, stdout=log_stream, stderr=subprocess.STDOUT, text=True)
print('started PID', process.pid, ' '.join(cmd))


In [ ]:
# Keep this cell running. It backs up resumable state every five minutes.
def progress_rows():
    rows=[]
    for path in LOCAL_ROOT.rglob('training_history.json'):
        try:
            history=json.loads(path.read_text(encoding='utf-8'))
            rows.append((str(path.parent.relative_to(LOCAL_ROOT)), int(history[-1]['iteration'])))
        except Exception:
            pass
    return sorted(rows)
try:
    while process.poll() is None:
        shutil.copytree(LOCAL_ROOT, DRIVE_ROOT, dirs_exist_ok=True)
        rows=progress_rows()
        print(time.strftime('%Y-%m-%d %H:%M:%S'), 'runs', len(rows), 'latest', rows[-8:])
        time.sleep(300)
finally:
    log_stream.close()
    shutil.copytree(LOCAL_ROOT, DRIVE_ROOT, dirs_exist_ok=True)
print('return code', process.returncode)
if process.returncode:
    print('\n'.join(log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-100:]))
    raise RuntimeError('Formal matrix failed; rerun cells 2-5 to resume')


## Recovery
If Colab disconnects, reconnect a GPU runtime and rerun all cells. The notebook restores the latest Drive backup and each job resumes from `resume_latest.pt`. Final artifacts remain under `MyDrive/GPPO_phase1_formal`.
